# 01b. Data quality checks

**Goal:** before I build any model, I want to know whether I can trust the data. A model built on faulty data gives faulty results, however good the model is.

**The questions I ask**
- Are any values missing, or hidden behind special codes such as `999`?
- Are there duplicate loans or duplicate rows?
- Do the dates make sense?
- Are the numbers in a believable range?
- Do the two files agree with each other?

**What I check, step by step**
1. **Origination file** (one row per loan): blanks and hidden codes in every column, duplicates, whether the loan identifier matches the file year, dates, number ranges, and logic (for example, combined loan-to-value should never be lower than loan-to-value).
2. **Performance file** (one row per loan per month): blanks in every column, repeated months, gaps or backward steps in the months, whether loan age goes up by one each month, whether the codes are valid, rows after a loan has ended, and whether balances behave.
3. **Both files together:** does every loan appear in both files, and do the starting balance and the first month agree?

Column positions (0, 1, 2, ...) are used here because the checks were written before the names were confirmed. The names are now in `src/columns.py` (taken from the user guide's attribute order).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
YEARS = list(range(2002, 2008))
# show all columns
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 70)

# dates are YYYYMM (200204), turn them into a month number so they can be subtracted
def to_idx(yyyymm):
    """YYYYMM -> a running month number, so months can be subtracted."""
    p = pd.to_numeric(yyyymm, errors="coerce")
    return (p // 100) * 12 + (p % 100)

# collects every check for the summary at the end
checks = []   # collects (area, check, count, comment) for the summary at the end
def record(area, check, count, comment=""):
    checks.append({"area": area, "check": check, "count": int(count), "comment": comment})

## A. Origination file (one row per loan)

In [2]:
# load all six years as text, blanks stay as empty strings
# columns are just numbered 0 to 30 here
orig = pd.concat(
    [pd.read_csv(RAW / f"sample_orig_{y}.txt", sep="|", header=None, dtype=str,
                 keep_default_na=False).assign(vintage=y) for y in YEARS],
    ignore_index=True)
orig.columns = list(range(31)) + ["vintage"]
print(f"{len(orig):,} loans, {orig.shape[1] - 1} columns")

300,000 loans, 31 columns


### A1. Every column: how many blanks, how many different values, which values are most common?

A human reads this table top to bottom. Blank columns, columns with one value, and columns with strange values all show up here.

In [3]:
# one line per column: blanks, distinct values, most common values
profile = pd.DataFrame({
    "blank_%": [round((orig[c].str.strip() == "").mean() * 100, 2) for c in range(31)],
    "distinct": [orig[c].nunique() for c in range(31)],
    "most_common (count)": [", ".join(f"{k!r}({v})" for k, v in orig[c].value_counts().head(4).items()) for c in range(31)],
})
profile.index.name = "col"
profile

,blank_%,distinct,most_common (count)
col,,,
0,0.0,383,"'782'(2534), '784'(2452), '766'(2436), '779'(2425)"
1,0.0,110,"'200406'(5833), '200309'(5638), '200405'(5504), '200505'(5368)"
2,0.0,3,"'N'(276843), 'Y'(23095), '9'(62)"
3,0.0,372,"'203704'(4177), '203801'(3908), '203504'(3842), '203604'(3833)"
4,34.7,453,"''(104111), '16974'(7180), '38060'(4394), '12060'(4356)"
5,0.0,35,"'0'(257412), '25'(16438), '30'(10395), '12'(7406)"
6,0.0,5,"'1'(293537), '2'(4996), '3'(756), '4'(704)"
7,0.0,3,"'P'(273919), 'I'(13403), 'S'(12678)"
8,0.0,124,"'80'(52428), '95'(19533), '90'(19114), '70'(9007)"


### A2. Duplicates and loan identifiers

In [4]:
# duplicates, blank ids, and does the year in the id match the file?
dup_rows = orig.duplicated(subset=list(range(31))).sum()
dup_ids = orig[19].duplicated().sum()
blank_ids = (orig[19].str.strip() == "").sum()
id_year = 2000 + pd.to_numeric(orig[19].str[1:3], errors="coerce")
id_mismatch = (id_year != orig["vintage"]).sum()
record("orig", "fully duplicated rows", dup_rows)
record("orig", "duplicated loan identifiers", dup_ids)
record("orig", "blank loan identifiers", blank_ids)
record("orig", "loan identifier year differs from file year", id_mismatch)
pd.DataFrame({"count": {"duplicated rows": dup_rows, "duplicated loan ids": dup_ids,
                        "blank loan ids": blank_ids, "id year differs from file year": id_mismatch}})

,count
duplicated rows,0
duplicated loan ids,0
blank loan ids,0
id year differs from file year,0


### A3. Dates: first payment date and maturity date

The first payment should come shortly after the loan starts (same year as the loan year, give or take a few months). The time from first payment to maturity should match the loan term.

In [5]:
# dates: first payment to maturity should match the loan term
fp = to_idx(orig[1])
mat = to_idx(orig[3])
term = pd.to_numeric(orig[21], errors="coerce")
fp_year = (pd.to_numeric(orig[1], errors="coerce") // 100)
# month must be 1 to 12
bad_month = ((pd.to_numeric(orig[1], errors="coerce") % 100).between(1, 12) == False).sum()
span = mat - fp
# allow 1 month difference
term_gap = (span - (term - 1)).abs() > 1
record("orig", "first payment date has an impossible month", bad_month)
record("orig", "maturity not consistent with loan term (off by more than 1 month)", term_gap.sum())
record("orig", "maturity date before first payment", (mat <= fp).sum())
# year of first payment by loan year
print("Year of first payment, by file year (rows = file year):")
pd.crosstab(orig["vintage"], fp_year)

Year of first payment, by file year (rows = file year):


1,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2014
vintage,,,,,,,,,,,,
2002,42085,7885,23,2,2,0,2,1,0,0,0,0
2003,0,42515,7449,24,1,2,6,1,0,1,1,0
2004,0,0,41094,8832,45,9,4,14,0,1,0,1
2005,0,0,0,41986,7964,33,7,6,3,1,0,0
2006,0,0,0,0,41429,8533,21,13,0,1,3,0
2007,0,0,0,0,0,41429,8527,36,5,2,1,0


### A4. Numbers: are they in a believable range, and do they agree with each other?

In [6]:
# text to numbers
score = pd.to_numeric(orig[0], errors="coerce")
cltv = pd.to_numeric(orig[8], errors="coerce")
dti = pd.to_numeric(orig[9], errors="coerce")
upb = pd.to_numeric(orig[10], errors="coerce")
ltv = pd.to_numeric(orig[11], errors="coerce")
rate = pd.to_numeric(orig[12], errors="coerce")

# anything unreadable (should be zero)
n_not_numeric = {name: int(s.isna().sum()) for name, s in
                 {"credit score": score, "combined LTV": cltv, "DTI": dti,
                  "balance": upb, "LTV": ltv, "rate": rate, "term": term}.items()}
# range checks, the codes are counted separately
# combined LTV should never be below LTV
res = {
    "credit score not 300-850 (excluding 9999)": int((((score < 300) | (score > 850)) & (score != 9999)).sum()),
    "credit score = 9999 (code)": int((score == 9999).sum()),
    "LTV not 1-200 (excluding 999)": int((((ltv < 1) | (ltv > 200)) & (ltv != 999)).sum()),
    "LTV = 999 (code)": int((ltv == 999).sum()),
    "combined LTV smaller than LTV": int(((cltv < ltv) & (cltv != 999) & (ltv != 999)).sum()),
    "DTI above 65 (excluding 999)": int(((dti > 65) & (dti != 999)).sum()),
    "DTI = 999 (code)": int((dti == 999).sum()),
    "balance zero or negative": int((upb <= 0).sum()),
    "interest rate not between 0 and 15": int(((rate <= 0) | (rate > 15)).sum()),
    "loan term not between 12 and 480 months": int(((term < 12) | (term > 480)).sum()),
}
for k, v in res.items():
    record("orig", k, v)
print("Values that could not be read as numbers:", n_not_numeric)
pd.DataFrame({"loans": res})

Values that could not be read as numbers: {'credit score': 0, 'combined LTV': 0, 'DTI': 0, 'balance': 0, 'LTV': 0, 'rate': 0, 'term': 0}


,loans
credit score not 300-850 (excluding 9999),0
credit score = 9999 (code),597
LTV not 1-200 (excluding 999),0
LTV = 999 (code),22
combined LTV smaller than LTV,0
DTI above 65 (excluding 999),0
DTI = 999 (code),6596
balance zero or negative,0
interest rate not between 0 and 15,0
loan term not between 12 and 480 months,0


### A5. Places: are the states real?

In [7]:
# state must be a real US state or territory
valid_states = set("AL AK AZ AR CA CO CT DE DC FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY PR GU VI".split())
bad_state = orig.loc[~orig[16].isin(valid_states), 16].value_counts()
record("orig", "state not a US state/territory code", bad_state.sum())
print("Unrecognised state values:", dict(bad_state) if len(bad_state) else "none")
print()
# California etc. should be at the top
print("Loans per state (top 8):")
orig[16].value_counts().head(8).to_frame("loans").T

Unrecognised state values: none

Loans per state (top 8):


16,CA,FL,IL,TX,MI,OH,NY,PA
loans,29438,19505,16249,16053,12931,12155,11747,11316


## B. Performance file (one row per loan per month)

Each year is read in two passes: one pass over **all 35 columns** to count blanks, and one pass over the six columns I use (loan identifier, month, balance, delinquency status, loan age, zero balance code) to check the logic.

In [8]:
# valid delinquency codes: 00-99, RA and XX
VALID_DQ = {f"{i:02d}" for i in range(100)} | {"RA", "XX"}

# all performance checks for one year, so only one year is in memory
def perf_year(year, orig_y):
    f = RAW / f"sample_perf_{year}.txt"

    # pass 1: blanks in all 35 columns
    blanks = np.zeros(35, dtype="int64")
    n = 0
    for ch in pd.read_csv(f, sep="|", header=None, dtype=str, keep_default_na=False, chunksize=500_000):
        n += len(ch)
        blanks += (ch.to_numpy(dtype=object) == "").sum(axis=0)

    # pass 2: the six columns I use
    d = pd.read_csv(f, sep="|", header=None, usecols=[0, 1, 2, 3, 4, 8], dtype=str, keep_default_na=False)
    d.columns = ["id", "period", "upb", "dq", "age", "zb"]
    d["idx"] = to_idx(d["period"])
    d["upb_n"] = pd.to_numeric(d["upb"], errors="coerce")
    d["age_n"] = pd.to_numeric(d["age"], errors="coerce")

    # compare a row with the one above, only if it is the same loan
    same = (d["id"] == d["id"].shift()).fillna(False)          # same loan as the row above
    next_same = (d["id"] == d["id"].shift(-1)).fillna(False)   # same loan as the row below
    loans = d["id"].nunique()
    contiguous = int((~same).sum()) == loans                   # each loan's rows sit together

    # month step inside a loan, and rows where the loan ended
    step = d["idx"].diff()
    zb_present = d["zb"].str.strip() != ""
    ended = zb_present.groupby(d["id"], sort=False).max()
    first = d.groupby("id", sort=False).first()
    last = d.groupby("id", sort=False).last()
    # last month in the file = where the data end
    cutoff = int(d["idx"].max())
    y, m = divmod(cutoff - 1, 12)

    # link to the origination file
    # starting balance and first month should agree
    o = orig_y.set_index(19)
    orig_upb = pd.to_numeric(o[10], errors="coerce")
    fp_idx = to_idx(o[1])
    joined_upb = first["upb_n"].reindex(o.index)
    lag = (first["idx"].reindex(o.index) - fp_idx)

    # all checks for this year (0 = nothing wrong)
    out = {
        "year": year, "rows": n, "loans": loans,
        "each loan's rows together": contiguous,
        "duplicate (loan, month) rows": int(d.duplicated(["id", "period"]).sum()),
        "month gaps within a loan": int(((step > 1) & same).sum()),
        "month goes backwards / repeats": int(((step <= 0) & same).sum()),
        "loan age not +1 when month is +1": int(((d["age_n"].diff() != 1) & same & (step == 1)).sum()),
        "invalid delinquency codes": int((~d["dq"].isin(VALID_DQ)).sum()),
        "delinquency = XX (not available)": int((d["dq"] == "XX").sum()),
        "negative balance": int((d["upb_n"] < 0).sum()),
        "balance not readable": int(d["upb_n"].isna().sum()),
        "loans with 2+ zero-balance rows": int((zb_present.groupby(d["id"], sort=False).sum() > 1).sum()),
        "rows after the loan ended": int((zb_present & next_same).sum()),
        "ended-row balance is zero (%)": round(float((d.loc[zb_present, "upb_n"] == 0).mean() * 100), 1),
        "loans not ended that stop before the data end": int(((~ended.reindex(last.index)) & (last["idx"] < cutoff)).sum()),
        "data end (last month)": f"{y}{m + 1:02d}",
        "orig loans missing from perf": len(set(o.index) - set(first.index)),
        "perf loans missing from orig": len(set(first.index) - set(o.index)),
        "start balance differs from orig by > $1,000": int(((joined_upb - orig_upb).abs() > 1000).sum()),
        "first month not the first-payment month (+/-1)": int((lag.abs() > 1).sum()),
    }
    # loan age at the first row, normally 0
    first_age = first["age_n"].value_counts().head(4).to_dict()
    return out, blanks / n * 100, first_age

In [9]:
# run all six years
results, blank_tables, first_ages = [], {}, {}
for y in YEARS:
    out, blank_pct, fa = perf_year(y, orig[orig["vintage"] == y])
    results.append(out)
    blank_tables[y] = blank_pct
    first_ages[y] = fa
    print(y, "done", flush=True)
perf_res = pd.DataFrame(results).set_index("year")
perf_res.T

2002 done


2003 done


2004 done


2005 done


2006 done


2007 done


year,2002,2003,2004,2005,2006,2007
rows,2497266,4080343,3988198,3880689,3206738,3015985
loans,50000,50000,50000,50000,50000,50000
each loan's rows together,True,True,True,True,True,True
"duplicate (loan, month) rows",0,0,0,0,0,0
month gaps within a loan,4,8,22,30,32,30
month goes backwards / repeats,0,0,0,0,0,0
loan age not +1 when month is +1,446,700,1306,2315,3306,3886
invalid delinquency codes,0,0,0,0,0,0
delinquency = XX (not available),0,0,0,0,0,0
negative balance,0,0,0,0,0,0


### B1. Blanks in every performance column (% of rows)

Only columns that are blank at least once are shown. A column that is blank in 100% of rows carries no information, and one that is blank most of the time is probably only filled in for special events (for example, the costs of a foreclosure).

In [10]:
# % blank per performance column, by year
blank_df = pd.DataFrame(blank_tables).round(2)
blank_df.index.name = "col"
blank_df[blank_df.max(axis=1) > 0]

,2002,2003,2004,2005,2006,2007
col,,,,,,
6,99.99,100.00,100.00,99.99,99.98,99.97
7,98.95,98.96,98.00,96.34,93.69,92.23
8,98.01,98.79,98.77,98.74,98.47,98.38
9,98.01,98.79,98.77,98.74,98.47,98.38
12,99.90,99.93,99.87,99.82,99.76,99.72
13,99.98,99.98,99.96,99.93,99.88,99.87
14,99.98,99.98,99.96,99.93,99.88,99.87
15,99.98,99.98,99.96,99.93,99.88,99.87
16,99.98,99.98,99.96,99.93,99.88,99.87


### B2. At which loan age does each loan first appear?

In [11]:
# first loan age seen for each loan
pd.DataFrame(first_ages).T.fillna(0).astype(int).rename_axis("year")

,0,1,2,3
year,,,,
2002,41576,5478,1306,465
2003,38881,7384,1862,605
2004,42214,5028,1089,443
2005,43168,4253,930,378
2006,41460,4367,1217,588
2007,42091,4103,1101,630


In [12]:
# add the performance checks to the summary
for k in ["duplicate (loan, month) rows", "month gaps within a loan", "month goes backwards / repeats",
          "loan age not +1 when month is +1", "invalid delinquency codes", "negative balance",
          "balance not readable", "loans with 2+ zero-balance rows", "rows after the loan ended",
          "loans not ended that stop before the data end", "orig loans missing from perf",
          "perf loans missing from orig", "start balance differs from orig by > $1,000",
          "first month not the first-payment month (+/-1)", "delinquency = XX (not available)"]:
    record("perf", k + " (all years)", perf_res[k].sum())
record("perf", "loans whose rows are NOT kept together in the file", (~perf_res["each loan's rows together"]).sum())

## C. Summary of all checks

In [13]:
# 0 is fine, above 0 needs a look
summary = pd.DataFrame(checks)
summary["verdict"] = np.where(summary["count"] == 0, "OK", "LOOK AT THIS")
summary[["area", "check", "count", "verdict"]]

,area,check,count,verdict
0,orig,fully duplicated rows,0,OK
1,orig,duplicated loan identifiers,0,OK
2,orig,blank loan identifiers,0,OK
3,orig,loan identifier year differs from file year,0,OK
4,orig,first payment date has an impossible month,0,OK
5,orig,maturity not consistent with loan term (off by more than 1 month),0,OK
6,orig,maturity date before first payment,0,OK
7,orig,credit score not 300-850 (excluding 9999),0,OK
8,orig,credit score = 9999 (code),597,LOOK AT THIS
9,orig,LTV not 1-200 (excluding 999),0,OK


## D. What I see

**Good news (these checks all passed)**
- No duplicate loans or rows, no blank loan identifiers, and the loan identifier always matches the file year.
- Every loan appears in both files (nothing missing either way), and rows are never repeated or going backwards in time.
- Dates make sense: the first payment is almost always in the loan year or the year after (a handful are later, see below), and the maturity date matches the loan term for every loan.
- Numbers are in believable ranges: no negative or zero balances, no impossible interest rates or terms, and all states are real US states or territories. **A note on three of these checks:** Freddie Mac replaces out-of-range values with a "Not Available" code before it publishes the data (a credit score outside 300-850, a debt-to-income ratio above 65%, a combined loan-to-value below the loan-to-value). So for those three the check cannot fail by design, and passing it is not independent proof that the values are right.
- Delinquency codes are all valid, and every loan has at most one "ended" row, which is always its last row, with a zero balance.
- Every loan that has not ended is still being reported at the end of the data (June 2026). No loans just disappear.

**Hidden "not available" codes (not blanks, so they hide from a normal null check)**
- Credit score `9999`: 597 loans. Debt-to-income `999`: 6,596 loans (2.2%). Loan-to-value `999`: 22 loans.
- First-time-homebuyer flag `9`: 62 loans. Number of borrowers `99`: 90 loans. The user guide confirms that all of these codes mean "Not Available".
- Plan: turn these codes into proper missing values before modelling.

**Columns that carry no information**
- Columns 25 and 26 are blank in every row. Columns 15, 24, 27, 28, 29 and 30 hold the same single value for all 300,000 loans (for example, every loan is fixed rate). I can drop all eight.
- Column 4 (a metropolitan area code) is blank for about 35% of loans. This is probably rural areas, but I have not confirmed it.
- In the performance files several columns are almost always blank. They are probably only filled in for special events (modification, foreclosure costs), which is expected.

**Things that look odd (small, but I should understand them before I trust the default rule)**
- A few loans have a first payment date several years after the loan year (for example, a 2002 loan with a first payment in 2008, and one 2004 loan with 2014). It is only a handful of loans, but it is strange.
- Loan age does not always go up by one when the month goes up by one: 11,959 of about 20 million rows (0.06%), more in later years. **Checked for 2004 and 2007:** the cause is loan modification. Among loans with such a jump, 93% (2004) and 97% (2007) carry a modification flag, and loans without a jump carry none. This matches the user guide, which says loan age restarts after a modification. A few loans (3% to 7%) have a jump without a modification flag, and I have not explained those.
- There are 126 gaps of a month or more in loan histories (in total, across all years). I have not looked at what causes them.
- For about 3.5% of loans the starting balance in the performance file differs from the original balance by more than $1,000, and for about 4% the first performance month is not the first payment month. Between about 14% and 22% of loans (depending on the year) first appear at loan age 1 or more instead of 0. **Checked for 2004 and 2007:** these are mostly the same loans. When a loan first appears at age 0, a balance mismatch is rare (about 0.4%). When it first appears later, the mismatch is common (18% to 21%) and so is a first month that is off (21% to 34%). So the most likely reason is that these loans enter the file after a few months, by which time the balance has already been paid down. The rounding the user guide mentions (to the nearest $1,000) cannot explain differences above $500, so it is not the reason.

**What this means for my project**
- A first version of the default rule counted defaults by **loan age**, which restarts after a modification. I then tested a second rule based on calendar months since the first payment (notebook 02). It gives clearly lower default rates (for example 2005: 2.13% against 1.05%), so the calendar rule is the one used from now on.
- About 14% to 22% of loans first show up at loan age 1 or more. A default needs at least three missed payments, so a loan first seen at age 1 to 3 cannot have defaulted before it appears. A small group first appears at an older age than that, and for those the early months are missing. I have not checked how many.
- Missing values must be handled on purpose. LightGBM can use blanks as they are, but logistic regression needs them filled or flagged.

**Not checked yet**
- Outliers within the performance files (for example, sudden jumps in balance).
- Whether the 50,000-loan sample is representative of all Freddie Mac loans of that year.
- Column names are now taken from the user guide's attribute order (see `src/columns.py`). The separate technical file layout (field types and positions) has not been downloaded.